# Steam market analysis for Ubisoft: 01 Macro analysis
Publishers, ratings, releases over time (including Covid), prices and discounts, languages, age restrictions.

Charts are Databricks visualizations attached to the `display()` cells. If a chart is missing after import, the cell description gives its settings (**+ > Visualization**).

In [0]:
%run ./setup_data

# Steam analysis: shared data setup
Loads and cleans the Steam catalog and builds the dataframes used by every analysis notebook. It is executed by the other notebooks with `%run ./setup_data` and displays nothing on purpose (lighter published notebooks). The exploration of the raw data and the justification of each cleaning step are in `00_data_exploration`.

| Dataframe | Grain | Content |
|---|---|---|
| `games_df` | 1 row per game | cleaned columns (prices, dates, owners, reviews, ages, platforms) |
| `genres_df` | 1 row per (game, genre) | `genre` exploded |
| `languages_df` | 1 row per (game, language) | `language` exploded |
| `platforms_df` | 1 row per (game, platform) | Windows / Mac / Linux |
| `tags_df` | 1 row per (game, tag) | community tags and their vote count |

### Cleaning and feature engineering
- **prices** are strings in cents: converted to USD (`try_cast` returns null instead of failing on odd values, which matters on Spark 4 / serverless where ANSI mode is on).
- **release_date** comes as `yyyy/M/d` or only `yyyy/M`: parsed to a date, `release_year` and `release_month` extracted.
- **owners** is a range (`"20,000 .. 50,000"`): lower bound, upper bound and midpoint kept. It is an estimate of sales volume (SteamSpy).
- **reviews**: total, positive ratio and the Wilson lower bound of the positive ratio (ranks games by rating while penalising small review counts).
- **required_age**: first number extracted (`"MA 15+"` becomes 15).
- **estimated_revenue_usd** = owners midpoint x current price: a rough order of magnitude, not actual sales (prices change, discounts, regional pricing).
- Only `type == "game"` is kept; duplicated `appid` are removed.

## 1. Which publishers released the most games?
*Chart: bar, X = publisher, Y = games.*

In [0]:
top_publishers_df = (games_df.filter(F.col("publisher").isNotNull())
    .groupBy("publisher")
    .agg(F.count("*").alias("games"), F.round(F.sum("owners_mid") / 1e6, 1).alias("estimated_owners_millions"),
         F.round(F.avg("positive_ratio") * 100, 1).alias("avg_positive_pct"))
    .orderBy(F.desc("games")).limit(20))
display(top_publishers_df)

publisher,games,estimated_owners_millions,avg_positive_pct
Big Fish Games,423,5.2,83.8
8floor,202,2.0,74.6
SEGA,165,92.3,85.6
Strategy First,151,7.8,56.2
Square Enix,141,77.6,75.2
Choice of Games,140,1.5,74.0
Sekai Project,132,8.5,86.5
HH-Games,132,1.4,72.6
Ubisoft,127,172.6,77.6
Laush Studio,126,2.7,78.5


Databricks visualization. Run in Databricks to view.

In [0]:
n_publishers = games_df.filter(F.col("publisher").isNotNull()).select("publisher").distinct().count()
single = games_df.filter(F.col("publisher").isNotNull()).groupBy("publisher").count().filter("count = 1").count()
top20_share = top_publishers_df.agg(F.sum("games")).first()[0] / games_df.count()
print(f"{n_publishers:,} distinct publishers; {single:,} ({single / n_publishers:.0%}) published a single game; the top 20 publishers account for {top20_share:.1%} of all games.")

29,833 distinct publishers; 23,028 (77%) published a single game; the top 20 publishers account for 4.9% of all games.


*Chart: bar, X = publisher, Y = estimated_owners_millions.* Volume of games and audience are two different things:

In [0]:
top_publishers_owners_df = (games_df.filter(F.col("publisher").isNotNull())
    .groupBy("publisher").agg(F.round(F.sum("owners_mid") / 1e6, 1).alias("estimated_owners_millions"), F.count("*").alias("games"))
    .orderBy(F.desc("estimated_owners_millions")).limit(20))
display(top_publishers_owners_df)

publisher,estimated_owners_millions,games
Valve,802.9,35
Ubisoft,172.6,127
Electronic Arts,153.1,88
Amazon Games,113.5,3
Bethesda Softworks,112.3,55
Rockstar Games,98.3,19
SEGA,92.3,165
Xbox Game Studios,90.3,47
2K,78.8,67
Square Enix,77.6,141


Databricks visualization. Run in Databricks to view.

## 2. What are the best rated games?
A raw positive ratio favours games with 3 reviews and 100% positive. Games are ranked by the **Wilson lower bound** (positive ratio corrected by the number of reviews), among games with at least 1,000 reviews.

In [0]:
best_rated_df = (games_df.filter(F.col("total_reviews") >= 1000)
    .select("name", "publisher", "release_year", "total_reviews", F.round(F.col("positive_ratio") * 100, 2).alias("positive_pct"),
            F.round("wilson_score", 4).alias("wilson_score"), "owners")
    .orderBy(F.desc("wilson_score")).limit(20))
display(best_rated_df)

name,publisher,release_year,total_reviews,positive_pct,wilson_score,owners
Aseprite,Igara Studio,2016,11903,99.33,0.9916,"500,000 .. 1,000,000"
A Short Hike,adamgryu,2019,11732,99.26,0.9909,"100,000 .. 200,000"
Senren＊Banka,"HIKARI FIELD, NekoNyan Ltd.",2020,10677,99.21,0.9903,"100,000 .. 200,000"
Aventura Copilului Albastru și Urât,Codrin Bradea,2021,2217,99.37,0.9894,"50,000 .. 100,000"
People Playground,Studio Minus,2019,144569,98.86,0.988,"2,000,000 .. 5,000,000"
Portal 2,Valve,2011,309441,98.78,0.9874,"10,000,000 .. 20,000,000"
CULTIC,3D Realms,2022,2037,99.21,0.9873,"50,000 .. 100,000"
Vampire Survivors,poncle,2022,131935,98.77,0.9871,"2,000,000 .. 5,000,000"
Patrick's Parabox,Patrick Traynor,2022,1500,99.27,0.9869,"20,000 .. 50,000"
Ib,PLAYISM,2022,1829,99.18,0.9865,"20,000 .. 50,000"


*Chart: bar, X = positive_pct_bucket, Y = games.* Distribution of ratings (games with at least 50 reviews):

In [0]:
rating_dist_df = (games_df.filter(F.col("total_reviews") >= 50)
    .withColumn("positive_pct_bucket", (F.floor(F.col("positive_ratio") * 10) * 10).cast("int"))
    .withColumn("positive_pct_bucket", F.when(F.col("positive_pct_bucket") == 100, 90).otherwise(F.col("positive_pct_bucket")))
    .groupBy("positive_pct_bucket").agg(F.count("*").alias("games")).orderBy("positive_pct_bucket"))
display(rating_dist_df)

positive_pct_bucket,games
0,6
10,67
20,197
30,467
40,893
50,1683
60,2757
70,4322
80,5947
90,5329


Databricks visualization. Run in Databricks to view.

## 3. Are there years with more releases? What happened during Covid?
*Chart: line, X = release_year, Y = games.*

In [0]:
releases_year_df = (games_df.filter(F.col("release_year").between(2000, 2022))
    .groupBy("release_year").agg(F.count("*").alias("games")).orderBy("release_year"))
display(releases_year_df)
print("Latest release date in the dataset:", games_df.agg(F.max("release_date_clean")).first()[0], "-> the last year is partial")

release_year,games
2000,2
2001,4
2002,1
2003,3
2004,6
2005,6
2006,61
2007,98
2008,159
2009,311


Databricks visualization. Run in Databricks to view.

Latest release date in the dataset: 2022-11-11 -> the last year is partial


*Chart: line, X = release_month, Y = games.* Zoom on 2018-2022 by month (Covid lockdowns start in March 2020):

In [0]:
releases_month_df = (games_df.filter(F.col("release_month").between("2018-01-01", "2022-12-31"))
    .filter(F.length("release_date") >= 8)  # exclude dates given without day
    .groupBy("release_month").agg(F.count("*").alias("games")).orderBy("release_month"))
display(releases_month_df)

release_month,games
2018-01-01,501
2018-02-01,617
2018-03-01,745
2018-04-01,663
2018-05-01,683
2018-06-01,594
2018-07-01,644
2018-08-01,652
2018-09-01,652
2018-10-01,626


Databricks visualization. Run in Databricks to view.

In [0]:
w = Window.orderBy("release_year")
covid_df = (releases_year_df.filter(F.col("release_year").between(2016, 2022))
    .withColumn("growth_vs_previous_year_pct", F.round((F.col("games") / F.lag("games").over(w) - 1) * 100, 1)))
display(covid_df)
g = {r["release_year"]: r["growth_vs_previous_year_pct"] for r in covid_df.collect()}
print(f"Release growth: 2019 {g.get(2019)}%, 2020 {g.get(2020)}%, 2021 {g.get(2021)}% (2022 is incomplete)")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


release_year,games,growth_vs_previous_year_pct
2016,4185,null
2017,6017,43.8
2018,7678,27.6
2019,6968,-9.2
2020,8305,19.2
2021,8823,6.2
2022,7455,-15.5


Release growth: 2019 -9.2%, 2020 19.2%, 2021 6.2% (2022 is incomplete)


## 4. How are prices distributed? Are there many discounted games?
*Chart: bar, X = price_band, Y = games.*

In [0]:
price_band = (F.when(F.col("price_usd") == 0, "0. Free").when(F.col("price_usd") < 5, "1. < $5").when(F.col("price_usd") < 10, "2. $5-10")
              .when(F.col("price_usd") < 20, "3. $10-20").when(F.col("price_usd") < 40, "4. $20-40").otherwise("5. >= $40"))
games_df = games_df.withColumn("price_band", F.when(F.col("price_usd").isNotNull(), price_band))
price_dist_df = games_df.filter(F.col("price_band").isNotNull()).groupBy("price_band").agg(F.count("*").alias("games")).orderBy("price_band")
display(price_dist_df)
display(games_df.filter(F.col("price_usd") > 0).select(
    F.round(F.avg("price_usd"), 2).alias("avg_price_paid_games"),
    F.percentile_approx("price_usd", 0.5).alias("median_price_paid_games"),
    F.percentile_approx("price_usd", 0.9).alias("p90_price_paid_games"),
    F.max("price_usd").alias("max_price")))

price_band,games
0. Free,7779
1. < $5,23478
2. $5-10,12450
3. $10-20,9022
4. $20-40,2394
5. >= $40,567


Databricks visualization. Run in Databricks to view.

avg_price_paid_games,median_price_paid_games,p90_price_paid_games,max_price
8.99,5.99,19.99,999.0


*Chart: bar, X = discount_band, Y = games.*

In [0]:
discount_band = (F.when(F.col("discount_pct") == 0, "0. No discount").when(F.col("discount_pct") < 25, "1. 1-24%")
                 .when(F.col("discount_pct") < 50, "2. 25-49%").when(F.col("discount_pct") < 75, "3. 50-74%").otherwise("4. >= 75%"))
discount_df = (games_df.filter(F.col("price_usd") > 0).withColumn("discount_band", discount_band)
    .groupBy("discount_band").agg(F.count("*").alias("games")).orderBy("discount_band"))
display(discount_df)
share = games_df.filter(F.col("price_usd") > 0).select(F.avg((F.col("discount_pct") > 0).cast("int"))).first()[0]
print(f"{share:.1%} of paid games were discounted at the time of the extraction.")

discount_band,games
0. No discount,45393
1. 1-24%,224
2. 25-49%,506
3. 50-74%,942
4. >= 75%,846


Databricks visualization. Run in Databricks to view.

5.3% of paid games were discounted at the time of the extraction.


## 5. Most represented languages
*Chart: bar, X = language, Y = games.*

In [0]:
n_games = games_df.count()
top_languages_df = (languages_df.groupBy("language").agg(F.countDistinct("appid").alias("games"))
    .withColumn("pct_of_games", F.round(F.col("games") / n_games * 100, 1)).orderBy(F.desc("games")).limit(20))
display(top_languages_df)

language,games,pct_of_games
English,55120,99.0
German,14021,25.2
French,13427,24.1
Russian,12922,23.2
Simplified Chinese,12782,23.0
Spanish - Spain,12234,22.0
Japanese,10368,18.6
Italian,9305,16.7
Portuguese - Brazil,6750,12.1
Korean,6601,11.9


Databricks visualization. Run in Databricks to view.

*Chart: bar, X = n_languages_bucket, Y = games.*

In [0]:
n_lang_df = (games_df.withColumn("n_languages_bucket",
        F.when(F.col("n_languages") <= 1, "a. 0-1").when(F.col("n_languages") <= 3, "b. 2-3").when(F.col("n_languages") <= 6, "c. 4-6")
         .when(F.col("n_languages") <= 10, "d. 7-10").otherwise("e. 11+"))
    .groupBy("n_languages_bucket").agg(F.count("*").alias("games")).orderBy("n_languages_bucket"))
display(n_lang_df)

n_languages_bucket,games
a. 0-1,29665
b. 2-3,10966
c. 4-6,5692
d. 7-10,4788
e. 11+,4579


Databricks visualization. Run in Databricks to view.

## 6. Are there many games prohibited for children under 16 / 18?
*Chart: bar, X = age_restriction, Y = games.*

In [0]:
age_df = (games_df.withColumn("age_restriction",
        F.when(F.col("required_age_int") == 0, "a. No restriction").when(F.col("required_age_int") < 16, "b. 1-15")
         .when(F.col("required_age_int") < 18, "c. 16-17").otherwise("d. 18+"))
    .groupBy("age_restriction").agg(F.count("*").alias("games"))
    .withColumn("pct", F.round(F.col("games") / n_games * 100, 2)).orderBy("age_restriction"))
display(age_df)

age_restriction,games,pct
a. No restriction,55029,98.81
b. 1-15,355,0.64
c. 16-17,76,0.14
d. 18+,230,0.41


Databricks visualization. Run in Databricks to view.

## Key figures (computed from the data)

In [0]:
top_pub = top_publishers_df.first()
top_lang = top_languages_df.collect()
peak = releases_year_df.filter(F.col("release_year") < 2022).orderBy(F.desc("games")).first()
free_share = games_df.select(F.avg(F.col("is_free").cast("int"))).first()[0]
age16 = games_df.select(F.avg((F.col("required_age_int") >= 16).cast("int"))).first()[0]
print(f"- {n_games:,} games, {n_publishers:,} publishers. Top publisher by number of games: {top_pub['publisher']} ({top_pub['games']} games).")
print(f"- Peak year (complete years): {peak['release_year']} with {peak['games']:,} releases. Growth 2020: {g.get(2020)}%, 2021: {g.get(2021)}%.")
print(f"- Free games: {free_share:.1%}. Discounted paid games: {share:.1%}.")
print(f"- Languages: {top_lang[0]['language']} ({top_lang[0]['pct_of_games']}% of games), then {top_lang[1]['language']} ({top_lang[1]['pct_of_games']}%), {top_lang[2]['language']} ({top_lang[2]['pct_of_games']}%).")
print(f"- Games restricted to 16+: {age16:.1%}.")

- 55,690 games, 29,833 publishers. Top publisher by number of games: Big Fish Games (423 games).
- Peak year (complete years): 2021 with 8,823 releases. Growth 2020: 19.2%, 2021: 6.2%.
- Free games: 14.0%. Discounted paid games: 5.3%.
- Languages: English (99.0% of games), then German (25.2%), French (24.1%).
- Games restricted to 16+: 0.5%.
